
for i in $(seq 200 100 1600); do
  BAG_SIZE="$i" bash scripts/run_full_pipeline_without_coroping_omba_diffrentnumber_opf_crops.sh \
    --output-dir "$PWD/outputs/qwen2_5_10cls_sam/imnet${i}"
done

In [6]:
# JSON subset extractor: keep only the given keys (supports dot paths for nested keys)
# Usage is shown in the following cells.

import json
import os
from typing import Any, Dict, Iterable, List, Tuple, Union, Optional

JSONType = Union[Dict[str, Any], List[Any], str, int, float, bool, None]


def _get_by_path(data: JSONType, parts: List[str]) -> Tuple[Any, bool]:
    cur: Any = data
    for p in parts:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return None, False
    return cur, True


def _set_by_path(out: Dict[str, Any], parts: List[str], value: Any) -> None:
    cur: Dict[str, Any] = out
    for p in parts[:-1]:
        if p not in cur or not isinstance(cur[p], dict):
            cur[p] = {}
        cur = cur[p]  # type: ignore[assignment]
    cur[parts[-1]] = value


def extract_json_keys(
    data_or_path: Union[str, os.PathLike, Dict[str, Any]],
    keys: Iterable[str],
    output_path: Union[str, os.PathLike, None] = None,
    output_dir: Union[str, os.PathLike, None] = None,
    output_filename: Optional[str] = None,
    ignore_missing: bool = True,
) -> Tuple[Dict[str, Any], List[str], Optional[str]]:
    """
    Build a subset of a JSON object that contains only the specified keys.

    - data_or_path: dict or path to a JSON file
    - keys: iterable of keys. Supports dot paths for nested keys, e.g., "a.b.c"
    - output_path: optional full path to write the subset JSON
    - output_dir: optional directory to write the subset JSON (cannot be used with output_path)
    - output_filename: optional filename to use when output_dir is provided (defaults to <input_basename>.subset.json or "subset.json")
    - ignore_missing: if False, raise on missing key; if True, return list of missing

    Returns: (subset_dict, missing_keys, saved_path or None)
    """
    if isinstance(data_or_path, (str, os.PathLike)):
        with open(data_or_path, "r", encoding="utf-8") as f:
            data: Dict[str, Any] = json.load(f)
        input_path_str: Optional[str] = str(data_or_path)
    else:
        data = data_or_path  # type: ignore[assignment]
        input_path_str = None

    if output_path is not None and output_dir is not None:
        raise ValueError("Provide either output_path or output_dir, not both.")

    # If only a directory is provided, construct the output path
    if output_path is None and output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        if output_filename:
            fname = output_filename
        else:
            if input_path_str is not None:
                base = os.path.basename(input_path_str)
                root, _ = os.path.splitext(base)
                fname = f"{root}.subset.json"
            else:
                fname = "subset.json"
        output_path = os.path.join(str(output_dir), fname)

    subset: Dict[str, Any] = {}
    missing: List[str] = []

    for k in keys:
        parts = [part for part in str(k).split(".") if part]
        value, found = _get_by_path(data, parts)
        if found:
            _set_by_path(subset, parts, value)
        else:
            missing.append(str(k))
            if not ignore_missing:
                raise KeyError(f"Key not found: {k}")

    saved_path: Optional[str] = None
    if output_path:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(subset, f, indent=2, ensure_ascii=False)
        saved_path = str(output_path)

    return subset, missing, saved_path


In [7]:
# Provide your inputs here and run this cell to extract the subset.
# 1) Set input_path to your JSON file
# 2) Provide the list of keys (dot paths supported)
# 3) Optionally set output_path OR output_dir (+ optional output_filename) to write the result to disk

# Example:
# input_path = "/path/to/input.json"
# keys = ["title", "meta.author", "data.items", "config.flags.debug"]
# output_path = None  # or "/path/to/output.json"
# output_dir = "/path/to/save/dir"  # if set, a file like <input>.subset.json will be created
# output_filename = None  # e.g., "filtered.json" (used only with output_dir)

# ---- EDIT BELOW ----
input_path = "/mnt/abka03/Projects/xl-vlms/outputs/imagenet_all_200bag_size/inference/concepts_to_images.json" # e.g., "/mnt/abka03/Projects/xl-vlms/sample.json"
keys = ["beaver", "bear", "cat", "dog", "horse", "hot dog", "rabbit", "tiger", "zebra"]         # e.g., ["a", "b.c", "d.e.f"]
output_path = "/mnt/abka03/Projects/xl-vlms/outputs/qwen2_5_10cls/imagenet_all_100bag_size/inference/concepts_to_images.json"
output_dir = None
output_filename = None
# ---- EDIT ABOVE ----

if input_path is None:
    print("Set input_path to your JSON file path and provide keys.")
else:
    subset, missing, saved_to = extract_json_keys(
        input_path,
        keys,
        output_path=output_path,
        output_dir=output_dir,
        output_filename=output_filename,
    )
    print("Subset JSON:\n")
    print(json.dumps(subset, indent=2, ensure_ascii=False))
    if missing:
        print("\nMissing keys:", missing)
    if saved_to:
        print(f"\nSaved to: {saved_to}")


Subset JSON:

{
  "beaver": [
    "train/platypus/n01873310_60957.JPEG",
    "train/platypus/n01873310_1247.JPEG",
    "train/otter/n02444819_9492.JPEG",
    "train/otter/n02444819_2578.JPEG",
    "train/beaver/n02363005_6466.JPEG",
    "train/beaver/n02363005_2061.JPEG",
    "train/beaver/n02363005_6314.JPEG",
    "train/beaver/n02363005_19909.JPEG",
    "train/beaver/n02363005_12050.JPEG",
    "train/beaver/n02363005_4931.JPEG",
    "train/beaver/n02363005_5664.JPEG",
    "train/beaver/n02363005_878.JPEG",
    "train/beaver/n02363005_5710.JPEG",
    "train/beaver/n02363005_35309.JPEG",
    "train/beaver/n02363005_4598.JPEG",
    "train/beaver/n02363005_635.JPEG",
    "train/beaver/n02363005_4930.JPEG",
    "train/beaver/n02363005_4090.JPEG",
    "train/beaver/n02363005_4141.JPEG",
    "train/beaver/n02363005_12680.JPEG",
    "train/beaver/n02363005_3658.JPEG",
    "train/beaver/n02363005_370.JPEG",
    "train/beaver/n02363005_22876.JPEG",
    "train/beaver/n02363005_1692.JPEG",
    "

In [ ]:
# Quick self-test (optional): run to validate behavior with an inline dict

sample = {
    "a": [1,2,3],
    "beaver": [10,11],
    "bear": [20],
    "cat": [30,31],
    "x": "not a list",
    "title": "demo"
}

# New specialized extraction for flat mapping string->list

def extract_list_values(
    data_or_path: Union[str, os.PathLike, Dict[str, Any]],
    keys: Iterable[str],
    require_list: bool = True,
    output_path: Union[str, os.PathLike, None] = None,
    output_dir: Union[str, os.PathLike, None] = None,
    output_filename: Optional[str] = None,
) -> Tuple[Dict[str, List[Any]], List[str], List[str], Optional[str]]:
    """Extract only the specified keys (flat, no dot paths) whose values are lists.

    Returns (subset, missing_keys, non_list_keys, saved_path)
    """
    if isinstance(data_or_path, (str, os.PathLike)):
        with open(data_or_path, "r", encoding="utf-8") as f:
            data: Dict[str, Any] = json.load(f)
        input_path_str: Optional[str] = str(data_or_path)
    else:
        data = data_or_path  # type: ignore[assignment]
        input_path_str = None

    if output_path is not None and output_dir is not None:
        raise ValueError("Provide either output_path or output_dir, not both.")

    if output_path is None and output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        if output_filename:
            fname = output_filename
        else:
            if input_path_str is not None:
                base = os.path.basename(input_path_str)
                root, _ = os.path.splitext(base)
                fname = f"{root}.lists_subset.json"
            else:
                fname = "lists_subset.json"
        output_path = os.path.join(str(output_dir), fname)

    subset: Dict[str, List[Any]] = {}
    missing: List[str] = []
    non_list: List[str] = []

    for k in keys:
        if k not in data:
            missing.append(k)
            continue
        val = data[k]
        if isinstance(val, list):
            subset[k] = val
        else:
            non_list.append(k)
            if require_list:
                continue
            subset[k] = [val]

    saved_path: Optional[str] = None
    if output_path:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(subset, f, indent=2, ensure_ascii=False)
        saved_path = str(output_path)

    return subset, missing, non_list, saved_path

# Demo of list extraction
keys_demo = ["beaver", "bear", "cat", "dog", "x"]
subset_lists, missing_lists, non_list_lists, saved_lists = extract_list_values(sample, keys_demo)
print("List subset demo:\n", json.dumps(subset_lists, indent=2))
print("Missing (demo):", missing_lists)
print("Non-list (demo):", non_list_lists)
print("Saved to (demo):", saved_lists)


In [2]:
crops_path ="/mnt/abka03/Projects/xl-vlms/outputs/dtd_concepts/qwen2_5/imnet500/inference/crops.json"

In [3]:
import json
import shutil
from pathlib import Path

def collect_train_paths(json_data):
    """Return a list of all JSON image keys that start with 'train/'."""
    train_paths = set()
    for concept, images in json_data.items():
        if not isinstance(images, dict):
            continue
        for rel_path in images.keys():
            if isinstance(rel_path, str) and rel_path.startswith("train/"):
                train_paths.add(rel_path)
    return sorted(train_paths)

def find_source_file(dataset_root: Path, rel_path: str):
    """
    Try to find the real file in dataset_root.
    Strategy:
      1) dataset_root / rel_path
      2) dataset_root / rel_path without leading 'train/'
      3) search by filename under dataset_root if unique
    """
    p1 = dataset_root / rel_path
    if p1.exists():
        return p1

    stripped = rel_path[len("train/"):]
    p2 = dataset_root / stripped
    if p2.exists():
        return p2

    # fallback: search by filename
    fname = Path(rel_path).name
    matches = list(dataset_root.rglob(fname))
    matches = [m for m in matches if m.is_file()]

    if len(matches) == 1:
        return matches[0]
    elif len(matches) > 1:
        print(f"[AMBIGUOUS] {rel_path} -> multiple matches: "
              f"{[str(m.relative_to(dataset_root)) for m in matches[:5]]} ...")
        return None
    else:
        print(f"[MISSING] {rel_path} not found under dataset root.")
        return None

def move_train_files(json_data, dataset_root, train_dir, dry_run=True):
    dataset_root = Path(dataset_root)
    train_dir = Path(train_dir)
    train_dir.mkdir(parents=True, exist_ok=True)

    train_paths = collect_train_paths(json_data)
    print(f"Found {len(train_paths)} train files in JSON.")

    moved, skipped = 0, 0

    for rel_path in train_paths:
        src = find_source_file(dataset_root, rel_path)
        if src is None:
            skipped += 1
            continue

        # destination preserves subdirs after "train/"
        subpath = Path(rel_path[len("train/"):])   # e.g. scaly/scaly_0153.jpg
        dst = train_dir / subpath
        dst.parent.mkdir(parents=True, exist_ok=True)

        if dst.exists():
            print(f"[SKIP] Destination already exists: {dst}")
            skipped += 1
            continue

        print(f"[MOVE] {src}  ->  {dst}")
        if not dry_run:
            shutil.move(str(src), str(dst))
        moved += 1

    print("\nDone.")
    print(f"Moved:   {moved}")
    print(f"Skipped: {skipped}")


In [9]:
freq = {k: (len(v) if isinstance(v, dict) else 0) for k, v in data.items()}
top100 = sorted(freq.items(), key=lambda x: x[1], reverse=True)

print("Top 100 concepts by #samples:\n")
for k, c in top100:
    print(f"{k:30s}  {c}")


Top 100 concepts by #samples:

pattern                         1794
texture                         1776
white                           1031
fabric                          865
black                           730
brown                           640
surface                         592
design                          580
abstract                        542
green                           530
blue                            484
color                           455
background                      422
textured                        411
natural                         402
yellow                          384
red                             384
dot                             371
beige                           371
material                        366
textile                         352
grid                            350
line                            350
stripe                          345
gray                            305
leaf                            299
square                        

In [16]:
dataset_root = "/mnt/abka03/xlvlm_data/dtd"   # root where train/val are mixed
train_dir    = "/mnt/abka03/xlvlm_data/dtd_split/train"  # where you want to gather train images


In [17]:
move_train_files(data, dataset_root, train_dir, dry_run=False)


Found 5513 train files in JSON.
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0002.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train/banded/banded_0002.jpg
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0004.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train/banded/banded_0004.jpg
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0005.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train/banded/banded_0005.jpg
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0006.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train/banded/banded_0006.jpg
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0008.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train/banded/banded_0008.jpg
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0009.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train/banded/banded_0009.jpg
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0010.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train/banded/banded_0010.jpg
[MOVE] /mnt/abka03/xlvlm_data/dtd/banded/banded_0011.jpg  ->  /mnt/abka03/xlvlm_data/dtd_split/train

In [6]:
dtd_similar_list = [
    "striped",
    "banded",
    "lined",
    "zigzagged",
    "crosshatched",
    "grid",
    "chequered",
    "woven",
    "interlaced",
    "knitted",
    "braided",
    "latticed",
    "fibrous",
    "organic",
    "bumpy",
    "ridged",
    "scaly",
    "grooved",
    "veined",
    "wrinkled",
    "cracked",
    "porous",
    "swirly",
    "marbled",
    "bubbly",
    "studded",
    "colorful",
    "dotted",
    "sprinkled",
    "flecked",
    "spotted",
    "blotchy"
]


In [7]:
import json
from pathlib import Path

json_path = Path("/mnt/abka03/Projects/xl-vlms/outputs/dtd_concepts/crops.json")
with open(json_path, "r") as f:
    data = json.load(f)

print("Original #concepts:", len(data))


Original #concepts: 3527


In [10]:
keep_set = set(dtd_similar_list)

filtered_data = {k: v for k, v in data.items() if k in keep_set}

print("Filtered #concepts:", len(filtered_data))
print("Missing concepts (not found in JSON):", sorted(keep_set - set(data.keys())))

out_path = json_path.with_name(json_path.stem + "_animal_attire_only.json")
with open(out_path, "w") as f:
    json.dump(filtered_data, f, indent=2)

print("Saved to:", out_path)


Filtered #concepts: 18
Missing concepts (not found in JSON): ['banded', 'blotchy', 'bubbly', 'chequered', 'crosshatched', 'fibrous', 'flecked', 'grooved', 'latticed', 'porous', 'scaly', 'swirly', 'veined', 'zigzagged']
Saved to: /mnt/abka03/Projects/xl-vlms/outputs/dtd_concepts/crops_animal_attire_only.json
